### Task 1.

Предположим, у нас есть набор признаков, которые вычисляются независимо от эксперимента. Используя эти признаки, нужно разбить объекты на страты так, чтобы дисперсия стратифицированного среднего была минимальна и доля каждой страты была не менее 5% от всех данных.

Данные разбиты на 2 части. Первая часть доступна для исследования по ссылке stratification_task_data_public.csv. Решение будет проверяться на второй части данных. Значения в столбцах
x1, ..., x10 — признаки, которые можно использовать для вычисления страт. Значения в столбце y — измерения, по которым будет вычисляться целевая метрика эксперимента.

Подходы формирования страт можно посмотреть в блокноте.

Дисперсия должна не превышать 50000.

Шаблон решения

In [ ]:
import pandas as pd
import numpy as np


def get_strats(df_features):
    """Возвращает страты объектов.

    :param df_features (pd.DataFrame): таблица с признаками x1,...,x10
    :return (list | np.array | pd.Series): список страт объектов размера len(df).
    """
    # YOUR_CODE_HERE

Пример

In [34]:
df = pd.read_csv('stratification_task_data_public.csv')
df_features = df.drop('y', axis=1)
df['strat'] = get_strats(df_features)
# пример вычисления дисперсии смотри в прикреплённом jupyter-notebook

**Решение**

In [139]:
import pandas as pd
import numpy as np


def get_strats(df_features):
    """Возвращает страты объектов.

    :param df_features (pd.DataFrame): таблица с признаками x1,...,x10
    :return (list | np.array | pd.Series): список страт объектов размера len(df).
    """
    conditions = [
    ((df_features['x2'] < 26) | (df_features['x2'] >= 35)) & (df_features.x10 <= 1) & (df_features['x5'] == df_features['x9']),
    ((df_features['x2'] < 26) | (df_features['x2'] >= 35)) & (df_features.x10 <= 1) & (df_features['x5'] != df_features['x9']),
    ((df_features['x2'] < 26) | (df_features['x2'] >= 35)) & (df_features.x10 > 1) & (df_features['x5'] != df_features['x9']),
    (df_features['x2'] >= 26) & (df_features['x2'] < 35) & (df_features.x10 == 0),
    (df_features['x2'] >= 26) & (df_features['x2'] < 35) & (df_features.x10.isin([1, 2])) & (df_features['x5'] == df_features['x9']),
    (df_features['x2'] >= 26) & (df_features['x2'] < 35) & (df_features.x10.isin([1, 2])) & (df_features['x5'] != df_features['x9']),
    (df_features['x2'] >= 26) & (df_features['x2'] < 35) & (df_features.x10 > 2) & (df_features['x5'] == df_features['x9']),
    (df_features['x2'] >= 26) & (df_features['x2'] < 35) & (df_features.x10 > 2) & (df_features['x5'] != df_features['x9'])
    ]

    choices = [1, 2, 3, 4, 5, 6, 7, 8]
    df_features['strat'] = np.select(conditions, choices, default=9)
    return df_features['strat']

def calc_strat_params(df):
    """Вычисляет стратифицированную дисперсию и минимальную долю страт."""
    strat_vars = df.groupby('strat')['y'].var()
    weights = df['strat'].value_counts(normalize=True)
    stratified_var = (strat_vars * weights).sum()
    min_part = df['strat'].value_counts(normalize=True).min()
    return stratified_var, min_part

In [141]:
df = pd.read_csv('stratification_task_data_public.csv')
df_features = df.drop('y', axis=1)
df['strat'] = get_strats(df_features)

In [142]:
stratified_var, min_part = calc_strat_params(df)
print(f"var={stratified_var:0.0f}, min_part={min_part*100:0.2f}%")

var=49674, min_part=6.29%


### Task 2.

Допустим, мы заранее определили множество клиентов, которые будут участвовать в эксперименте. Их страты известны. Нужно написать функцию, которая будет стратифицировано распределять их по группам.

Распределение по группам будем считать стратифицированным, если для каждой страты количество клиентов этой страты в группах отличаются не более, чем на 1.

Реализуйте функцию split_stratified.

Шаблон решения

In [ ]:
import numpy as np


def split_stratified(strats):
    """Распределяет объекты по группам (контрольная и экспериментальная).

    :param strats (np.array): массив с разбиением на страты.
    :return groups (np.array): массив из 0 и 1,
        0 - контрольная группа, 1 - экспериментальная.
    """
    # YOUR_CODE_HERE

Пример

In [ ]:
df = pd.DataFrame({'strat': [1, 2, 2, 2, 1, 1, 1, 3, 3]})
df['group'] = split_stratified(df['strat'].values)
# df = pd.DataFrame({
#   'strat': [1, 2, 2, 2, 1, 1, 1, 3, 3],
#   'group': [1, 0, 0, 1, 0, 0, 1, 0, 1]
# })

**Решение**